In [ ]:
import cv2
import numpy as np
import ipywidgets as widgets
from IPython.display import display
import threading
import queue
import mediapipe as mp
import time
import vosk, sounddevice as sd, json
import json
from datetime import datetime
import ffmpeg
from pathlib import Path
import pyttsx3


In [ ]:

BaseOptions = mp.tasks.BaseOptions
PoseLandmarker = mp.tasks.vision.PoseLandmarker
PoseLandmarkerOptions = mp.tasks.vision.PoseLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode
MODEL_PATH = "pose_landmarker_full.task"


POSE_CONNECTIONS = [
	(0,1),(1,2),(2,3),(3,7),(0,4),(4,5),(5,6),(6,8),(9,10),
	(11,12),(11,23),(12,24),(23,24),
	(11,13),(13,15),(15,17),(15,19),(15,21),(17,19),
	(12,14),(14,16),(16,18),(16,20),(16,22),(18,20),
	(23,25),(25,27),(27,29),(29,31),
	(24,26),(26,28),(28,30),(30,32),
	(27,31),(28,32),
]


tts_engine = pyttsx3.init()
tts_engine.setProperty('rate', 160)

def speak(text):
	threading.Thread(target=lambda: (tts_engine.say(text), tts_engine.runAndWait()), daemon=True).start()

def draw_pose(frame, landmarks):
    h, w = frame.shape[:2]


    for start_idx, end_idx in POSE_CONNECTIONS:
        p1 = landmarks[start_idx]
        p2 = landmarks[end_idx]

        x1 = int(p1.x * w)
        y1 = int(p1.y * h)

        x2 = int(p2.x * w)
        y2 = int(p2.y * h)

        cv2.line(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)


    for lm in landmarks:
        x = int(lm.x * w)
        y = int(lm.y * h)

        cv2.circle(frame, (x, y), 4, (0, 0, 255), -1)


options = PoseLandmarkerOptions(
	base_options=BaseOptions(model_asset_path=MODEL_PATH),
	running_mode=VisionRunningMode.VIDEO,
	num_poses=1,
)

# Do przekazywania obrazu używałem na telefonie aplikacji IP webcam na androidzie
STREAM_URL = 0 #"http://192.168.135.9:8080/video"

MAX_WIDTH = 900
JPEG_QUALITY = 50
FPS = 15
TIMEOUT = 200 # zatrzymuje kamerke po X sekundach

model = vosk.Model("vosk-model-small-pl-0.22")

def listen_command(duration=3, sr=16000):
	audio = sd.rec(int(duration*sr), samplerate=sr, channels=1, dtype='int16')
	sd.wait()
	rec = vosk.KaldiRecognizer(model, sr)
	rec.AcceptWaveform(audio.tobytes())
	return json.loads(rec.Result())["text"].lower()

# Analiza przysiadów:
def angle(a, b, c):
	a = np.array([a.x, a.y])
	b = np.array([b.x, b.y])
	c = np.array([c.x, c.y])
	ba, bc = a - b, c - b
	cos = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
	return np.degrees(np.arccos(np.clip(cos, -1.0, 1.0)))

squat_state = {"phase": "up", "reps": 0}

def analyze_squat(landmarks):
	left_knee = angle(landmarks[23], landmarks[25], landmarks[27])
	right_knee = angle(landmarks[24], landmarks[26], landmarks[28])
	knee = (left_knee + right_knee) / 2

	if squat_state["phase"] == "up" and knee < 90:
		squat_state["phase"] = "down"
	elif squat_state["phase"] == "down" and knee > 160:
		squat_state["phase"] = "up"
		squat_state["reps"] += 1

	good = knee < 90 or knee > 160
	return squat_state["reps"], knee, good

def render_stats(reps, knee, good, errors):
	err_html = "".join([
		f"<div style='color:#ff6b6b;font-size:12px;margin-top:3px'>⚠ pow. {e['rep']}: {e['error']} ({e['knee_angle']}°)</div>"
		for e in errors[-3:]
	])
	color = "#00ff88" if good else "#ff6b6b"
	phase_icon = "⬇" if squat_state["phase"] == "down" else "⬆"
	stats_html.value = f"""
	<div style='font-family:monospace;background:#0d0d1a;border-radius:12px;
	padding:12px 18px;margin-top:6px;border:1px solid #1e1e3a'>
		<div style='display:flex;gap:32px;align-items:center'>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>POWTÓRZENIA</div>
				<div style='color:#fff;font-size:28px;font-weight:bold'>{reps}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>KĄT KOLAN</div>
				<div style='color:{color};font-size:28px;font-weight:bold'>{knee:.0f}°</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FAZA</div>
				<div style='color:#aaa;font-size:24px'>{phase_icon}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FORMA</div>
				<div style='color:{color};font-size:16px;font-weight:bold'>{"✓ OK" if good else "✗ GŁĘBIEJ"}</div>
			</div>
		</div>
		{err_html}
	</div>"""


def draw_pose(frame, landmarks):
	h, w = frame.shape[:2]
	for s, e in POSE_CONNECTIONS:
		p1, p2 = landmarks[s], landmarks[e]
		cv2.line(frame, (int(p1.x*w), int(p1.y*h)), (int(p2.x*w), int(p2.y*h)), (0, 255, 136), 2)
	for lm in landmarks:
		cv2.circle(frame, (int(lm.x*w), int(lm.y*h)), 4, (255, 100, 100), -1)
	reps, knee, good = analyze_squat(landmarks, frame)
	render_stats(reps, knee, good, recording["stats"])


stop = threading.Event()
q = queue.Queue(maxsize=1)
squat_state = {"phase": "up", "reps": 0}
recording = {"active": False, "frames": [], "stats": []}
vosk_model = vosk.Model("vosk-model-small-pl-0.22")

img_widget = widgets.Image(format='jpeg', layout=widgets.Layout(
	border='2px solid #1a1a2e',
	border_radius='12px',
	width='100%'
))


def resize(frame):
	h, w = frame.shape[:2]
	if w > MAX_WIDTH:
		return cv2.resize(frame, (MAX_WIDTH, int(h * MAX_WIDTH / w)))
	return frame


def analyze(frame):
	frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
	frame = cv2.cvtColor(frame, cv2.COLOR_GRAY2BGR)
	return frame


def fetch():
	cap = cv2.VideoCapture(STREAM_URL)
	cap.set(cv2.CAP_PROP_FPS, FPS)
	with PoseLandmarker.create_from_options(options) as landmarker:
		while cap.isOpened() and not stop.is_set():
			ret, frame = cap.read()
			if not ret:
				break
			frame = cv2.flip(frame, 1)
			rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
			mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
			result = landmarker.detect_for_video(mp_image, int(time.time() * 1000))
			if result.pose_landmarks:
				draw_pose(frame, result.pose_landmarks[0])
			try:
				q.put_nowait(resize(frame))
			except queue.Full:
				pass
	cap.release()

status_html = widgets.HTML(value="")
stats_html = widgets.HTML(value="")
voice_html = widgets.HTML(value="")

def render_status(active):
	dot = "#00ff88" if active else "#ff4444"
	label = "● NAGRYWANIE" if active else "● CZUWANIE"
	status_html.value = f"""
	<div style='font-family:monospace;font-size:13px;color:{dot};
	background:#0d0d1a;padding:6px 14px;border-radius:8px;
	display:inline-block;letter-spacing:1px;margin-bottom:6px'>{label}</div>"""

def process():
	while not stop.is_set():
		try:
			frame = q.get(timeout=1)
			_, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])
			img_widget.value = buf.tobytes()
		except:
			pass

def render_stats(reps, knee, good, errors):
	err_html = "".join([
		f"<div style='color:#ff6b6b;font-size:12px;margin-top:3px'>⚠ pow. {e['rep']}: {e['error']} ({e['knee_angle']}°)</div>"
		for e in errors[-3:]
	])
	color = "#00ff88" if good else "#ff6b6b"
	phase_icon = "⬇" if squat_state["phase"] == "down" else "⬆"
	stats_html.value = f"""
	<div style='font-family:monospace;background:#0d0d1a;border-radius:12px;
	padding:12px 18px;margin-top:6px;border:1px solid #1e1e3a'>
		<div style='display:flex;gap:32px;align-items:center'>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>POWTÓRZENIA</div>
				<div style='color:#fff;font-size:28px;font-weight:bold'>{reps}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>KĄT KOLAN</div>
				<div style='color:{color};font-size:28px;font-weight:bold'>{knee:.0f}°</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FAZA</div>
				<div style='color:#aaa;font-size:24px'>{phase_icon}</div>
			</div>
			<div>
				<div style='color:#888;font-size:11px;letter-spacing:1px'>FORMA</div>
				<div style='color:{color};font-size:16px;font-weight:bold'>{"✓ OK" if good else "✗ GŁĘBIEJ"}</div>
			</div>
		</div>
		{err_html}
	</div>"""

def render_voice(partial, final=""):
	voice_html.value = f"""
	<div style='font-family:monospace;font-size:12px;background:#0d0d1a;
	padding:6px 14px;border-radius:8px;margin-top:4px;border:1px solid #1e1e3a'>
		<span style='color:#555'>🎤 </span>
		<span style='color:#aaa'>{partial}</span>
		{"<span style='color:#00ff88;margin-left:8px'>→ " + final + "</span>" if final else ""}
	</div>"""


def start():
	stop.clear()
	render_status(False)
	render_stats(0, 180, True, [])
	render_voice("gotowy...")
	display(widgets.VBox([
		status_html,
		img_widget,
		stats_html,
		voice_html,
	], layout=widgets.Layout(gap='4px')))
	threading.Thread(target=fetch, daemon=True).start()
	threading.Thread(target=process, daemon=True).start()
	threading.Thread(target=listen_loop, daemon=True).start()
	threading.Timer(TIMEOUT, stop.set).start()

def listen_loop():
	rec = vosk.KaldiRecognizer(vosk_model, 16000)
	with sd.RawInputStream(samplerate=16000, channels=1, dtype='int16') as stream:
		while not stop.is_set():
			data, _ = stream.read(4000)
			if rec.AcceptWaveform(bytes(data)):
				cmd = json.loads(rec.Result())["text"].lower()
				if cmd:
					render_voice("", cmd)
					if "start" in cmd:
						squat_state["reps"] = 0
						squat_state["phase"] = "up"
						recording.update({"active": True, "frames": [], "stats": []})
						render_status(True)
					elif "reset" in cmd:
						squat_state["reps"] = 0
					elif "stop" in cmd:
						stop.set()
						if recording["active"]:
							save_session()
						render_status(False)
			else:
				partial = json.loads(rec.PartialResult())["partial"]
				if partial:
					render_voice(partial)



recording = {"active": False, "frames": [], "stats": []}

def listen_loop():
	rec = vosk.KaldiRecognizer(model, 16000)
	with sd.RawInputStream(samplerate=16000, channels=1, dtype='int16') as stream:
		while not stop.is_set():
			data, _ = stream.read(4000)
			if rec.AcceptWaveform(bytes(data)):
				cmd = json.loads(rec.Result())["text"].lower()
				if cmd:
					print(f"[VOICE] {cmd}")
					if "start" in cmd:
						speak("start, zaczynamy")
						squat_state["reps"] = 0
						squat_state["phase"] = "up"
						recording["active"] = True
						recording["frames"] = []
						recording["stats"] = []
					elif "reset" in cmd:
						squat_state["reps"] = 0
					elif "stop" in cmd:
						speak(f"koniec, wykonałeś {squat_state['reps']} powtórzeń")
						stop.set()
						if recording["active"]:
							save_session()
			else:
				partial = json.loads(rec.PartialResult())["partial"]
				if partial:
					print(f"[VOICE] {partial}", end="\r")


def save_session():
	recording["active"] = False
	if not recording["frames"]:
		return
	ts = datetime.now().strftime("%Y%m%d_%H%M%S")
	Path("sessions").mkdir(exist_ok=True)
	path = f"sessions/session_{ts}.mp4"
	h, w = recording["frames"][0].shape[:2]
	proc = (
		ffmpeg
		.input('pipe:', format='rawvideo', pix_fmt='bgr24', s=f'{w}x{h}', r=FPS)
		.output(path, pix_fmt='yuv420p', vcodec='libx264')
		.overwrite_output()
		.run_async(pipe_stdin=True, quiet=True)
	)
	for f in recording["frames"]:
		proc.stdin.write(f.tobytes())
	proc.stdin.close()
	proc.wait()
	with open(f"sessions/stats_{ts}.json", "w") as file:
		json.dump({"reps": squat_state["reps"], "errors": recording["stats"]}, file, indent=2, ensure_ascii=False)
	render_voice("", f"zapisano sesję ({squat_state['reps']} powtórzeń)")

def analyze_squat(landmarks, frame=None):
	left_knee = angle(landmarks[23], landmarks[25], landmarks[27])
	right_knee = angle(landmarks[24], landmarks[26], landmarks[28])
	knee = (left_knee + right_knee) / 2
	if squat_state["phase"] == "up" and knee < 90:
		squat_state["phase"] = "down"
	elif squat_state["phase"] == "down" and knee > 160:
		squat_state["phase"] = "up"
		squat_state["reps"] += 1
		speak(f"powtórzenie {squat_state['reps']}")

	if not good and recording["active"]:
		recording["stats"].append({...})
		speak("głębiej")
	good = knee < 90 or knee > 160
	if not good and recording["active"]:
		recording["stats"].append({
			"rep": squat_state["reps"],
			"knee_angle": round(knee, 1),
			"error": "Za płytki przysiad"
		})
	if recording["active"] and frame is not None:
		recording["frames"].append(frame.copy())
	return squat_state["reps"], knee, good


LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:ReadDataFiles():model.cc:216) Silence phones 1:2:3:4:5:6:7:8:9:10
LOG (VoskAPI:RemoveOrphanNodes():nnet-nnet.cc:948) Removed 0 orphan nodes.
LOG (VoskAPI:RemoveOrphanComponents():nnet-nnet.cc:847) Removing 0 orphan components.
LOG (VoskAPI:ReadDataFiles():model.cc:248) Loading i-vector extractor from vosk-model-small-pl-0.22/ivector/final.ie
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:183) Computing derived variables for iVector extractor
LOG (VoskAPI:ComputeDerivedVars():ivector-extractor.cc:204) Done.
LOG (VoskAPI:ReadDataFiles():model.cc:282) Loading HCL and G from vosk-model-small-pl-0.22/graph/HCLr.fst vosk-model-small-pl-0.22/graph/Gr.fst
LOG (VoskAPI:ReadDataFiles():model.cc:308) Loading winfo vosk-model-small-pl-0.22/graph/phones/word_boundary.int
LOG (VoskAPI:ReadDataFiles():model.cc:213) Decoding params beam=10 max-active=3000 lattice-beam=2
LOG (VoskAPI:R

In [ ]:
start()

I0000 00:00:1781727891.161550  156462 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1781727891.163620  156484 gl_context.cc:385] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: AMD Radeon RX 7900 XT (radeonsi, navi31, LLVM 20.1.2, DRM 3.64, 6.18.7-76061807-generic)
W0000 00:00:1781727891.201478  156472 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1781727891.222869  156475 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[VOICE] tak
[VOICE] restart
[VOICE] start
[VOICE] stop


In [18]:
stop.set()

In [50]:
def playback_session(stats_path, video_path):
	with open(stats_path) as f:
		stats = json.load(f)
	
	error_reps = {e["rep"] for e in stats["errors"]}
	cap = cv2.VideoCapture(video_path)
	frames = []
	while cap.isOpened():
		ret, frame = cap.read()
		if not ret:
			break
		frames.append(frame)
	cap.release()

	playback_widget = widgets.Image(format='jpeg', layout=widgets.Layout(width='100%'))
	info_html = widgets.HTML()

	def show(idx):
		frame = frames[idx].copy()
		rep_num = round(idx / len(frames) * stats["reps"])
		if rep_num in error_reps:
			cv2.rectangle(frame, (0,0), (frame.shape[1], frame.shape[0]), (0,0,255), 6)
			err = next((e for e in stats["errors"] if e["rep"] == rep_num), None)
			if err:
				cv2.putText(frame, f"BLAD: {err['error']} ({err['knee_angle']})", (20,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)
		_, buf = cv2.imencode('.jpg', frame, [cv2.IMWRITE_JPEG_QUALITY, 75])
		playback_widget.value = buf.tobytes()
		info_html.value = f"<div style='font-family:monospace;color:#aaa'>klatka {idx+1}/{len(frames)} | powtórzenia: {stats['reps']} | błędy: {len(stats['errors'])}</div>"

	slider = widgets.IntSlider(min=0, max=len(frames)-1, step=1, layout=widgets.Layout(width='100%'))
	widgets.interact(show, idx=slider)
	display(widgets.VBox([playback_widget, info_html]))

In [51]:
import glob, os

def playback_latest():
	sessions = sorted(glob.glob("sessions/stats_*.json"))
	if not sessions:
		print("brak sesji")
		return
	stats_path = sessions[-1]
	video_path = stats_path.replace("stats_", "session_").replace(".json", ".mp4")
	playback_session(stats_path, video_path)

playback_latest()

interactive(children=(IntSlider(value=0, description='idx', layout=Layout(width='100%'), max=92), Output()), _…